In [ ]:
import numpy as np
import dill as pickle
import tensorflow as tf

In [ ]:
path_to_kin = "/home/zanardi/Codes/ML/ROMAr/romar/examples/database/rates/kin_fit.p"
path_to_rad = "/home/zanardi/Codes/ML/ROMAr/romar/examples/database/rates/rad_fit.p"

In [ ]:
with open(path_to_kin, "rb") as file:
  kin = pickle.load(file)
with open(path_to_rad, "rb") as file:
  rad = pickle.load(file)

In [ ]:
tf.nest.map_structure(np.shape, rad)

In [ ]:
rad.pop('T')
rad.pop('BFp')
rad.pop('FF')
rad["BB"].pop('plato_id')
rad["BF"].pop('plato_id')
rad["BB"].pop('equation')
rad["BF"].pop('equation')

In [ ]:
tf.nest.map_structure(np.shape, rad)

In [ ]:
tf.nest.map_structure(np.shape, kin)

In [ ]:
kin.pop('T')
kin.pop('EN')
kin.pop('EI')
for k in ("EXe",'EXh','Ie','Ih'):
  kin[k].pop('plato_id')
  kin[k].pop('equation')

In [ ]:
tf.nest.map_structure(np.shape, kin)

In [ ]:
rates = tf.nest.map_structure(np.squeeze, {**kin, **rad})
tf.nest.map_structure(np.shape, rates)

In [ ]:
rates["EXh"]["temp_ind"] = (0, "Th")
rates["EXe"]["temp_ind"] = (1, "Te")
rates["Ih"]["temp_ind"] = (0, "Th")
rates["Ie"]["temp_ind"] = (1, "Te")
rates["BF"]["temp_ind"] = (1, "Te")
rates["BB"]["temp_ind"] = None

In [ ]:
tf.nest.map_structure(np.shape, rates)

In [ ]:
for k in ("EXe",'EXh'):
  for k2 in ("A", "beta", "Ta"):
    rates[k]['values'][k2] = rates[k]['values'][k2][1:,1:]
for k in ("Ie",'Ih','BF'):
  for k2 in ("A", "beta", "Ta"):
    if k2 in rates[k]['values']:
      rates[k]['values'][k2] = rates[k]['values'][k2][1:]
rates['BB']['values'] = rates['BB']['values'][1:,1:]
rates['BB']['lambda'] = rates['BB']['lambda'][1:,1:]
rates['BF']['lambda'] = rates['BF']['lambda'][1:,1:]

In [ ]:
tf.nest.map_structure(np.shape, rates)

In [ ]:
rates["EXh"]["comp_ind"] = {
  "reac": (np.arange(2,32),),
  "prod": (np.arange(2,32),)
}
rates["EXe"]["comp_ind"] = {
  "reac": (np.arange(2,32),),
  "prod": (np.arange(2,32),)
}
rates["Ih"]["comp_ind"] = {
  "reac": (np.arange(2,32),),
  "prod": (np.arange(32,34),np.array([0]))
}
rates["Ie"]["comp_ind"] = {
  "reac": (np.arange(2,32),),
  "prod": (np.arange(32,34),np.array([0]))
}
rates["BF"]["comp_ind"] = None
rates["BB"]["comp_ind"] = None

In [ ]:
rates["BB"]["values"] = {
  "A": rates["BB"]["values"],
  "lambda": rates["BB"].pop("lambda")
}
rates["BF"]["values"]["lambda"] = rates["BF"].pop("lambda")

In [ ]:
tf.nest.map_structure(np.shape, rates)

In [ ]:
with open("./rates.p", "wb") as file:
  pickle.dump(rates, file)

In [ ]:
rates2 = {}
nb_comp = 32
for k1 in ('EXe', 'EXh', 'BB'):
  rates2[k1] = {}
  for k2, c2 in rates[k1]["values"].items():
    rates2[k1][k2] = np.zeros((nb_comp, nb_comp))
    rates2[k1][k2][:30,:30] = c2
for k1 in ('Ie', 'Ih', 'BF'):
  rates2[k1] = {}
  for k2, c2 in rates[k1]["values"].items():
    rates2[k1][k2] = np.zeros((nb_comp, nb_comp))
    rates2[k1][k2][:30,-2:] = c2

In [ ]:
tf.nest.map_structure(np.shape, rates2)

In [ ]:
with open("./rates2.p", "wb") as file:
  pickle.dump(rates2, file)